---
layout: post
types: hacks
title: Data Science Feature
description: My feature for my data science website!
toc: true
comments: true
---

# Frontend

In [ ]:
<script>
      // Tooltip toggle
  const helpBtn = document.getElementById('helpBtn');
  const tooltip = document.getElementById('tooltip');

  helpBtn.addEventListener('mouseenter', () => {
    tooltip.style.display = 'block';
  });

  helpBtn.addEventListener('mouseleave', () => {
    tooltip.style.display = 'none';
  });

  tooltip.addEventListener('mouseenter', () => {
    tooltip.style.display = 'block';
  });

  tooltip.addEventListener('mouseleave', () => {
    tooltip.style.display = 'none';
  });

    document.getElementById('travelForm').addEventListener('submit', async function(e) {
      e.preventDefault();

      const season = document.getElementById('season').value;
      const activity = document.getElementById('activity').value;
      const budget = document.getElementById('budget').value;
      const continent = document.getElementById('continent').value;

      const resultDiv = document.getElementById('result');
      resultDiv.textContent = "Loading recommendation...";

      try {
        const response = await fetch('http://localhost:8887/api/destination/recommend', {
          method: 'POST',
          headers: { 'Content-Type': 'application/json' },
          body: JSON.stringify({
              season: season.toLowerCase(),
              activity: activity.toLowerCase(),
              budget: budget.toLowerCase(),
              continent: continent.toLowerCase()
           })
        });

        if (!response.ok) throw new Error("Request failed");

        const result = await response.json();

        if (result.destination && result.activitySuggestion) {
        resultDiv.innerHTML = `
          <h3>🌍 Recommended Destination:</h3>
          <p>${result.destination}</p>
          <h3>🎯 Suggested Activity:</h3>
          <p>${result.activitySuggestion}</p>
        `;
      } else {
        resultDiv.textContent = `❌ No recommendation found.`;
      }
    } catch (err) {
      resultDiv.textContent = "❌ Error connecting to the server.";
      console.error(err);
    }
  });
</script>

# Backend API

In [ ]:
from flask import Blueprint, request, jsonify
from flask_restful import Api, Resource
from model.destination import Destination  # Import the DestinationModel class

destination_api = Blueprint('destination_api', __name__, url_prefix='/api/destination')
api = Api(destination_api)

class DestinationAPI:
    class _Recommend(Resource):
        def post(self):
            data = request.get_json()
            model = Destination.get_instance()
            recommendation = model.predict(data)
            return jsonify(recommendation)

    api.add_resource(_Recommend, '/recommend')

# Backend Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
import numpy as np
import random

class Destination:
    _instance = None

    @staticmethod
    def get_instance():
        if Destination._instance is None:
            Destination()
        return Destination._instance

    def __init__(self):
        if Destination._instance is not None:
            raise Exception("This class is a singleton!")
        else:
            Destination._instance = self
            self._load_model()

    def _load_model(self):
        # [season, activity, budget, continent, destination]
        self.data = [
            ['summer', 'beach', 'low', 'asia', 'bali'],
            ['winter', 'skiing', 'high', 'europe', 'switzerland'],
            ['autumn', 'hiking', 'medium', 'north america', 'colorado'],
            ['spring', 'museums', 'medium', 'europe', 'paris'],
            ['summer', 'safari', 'high', 'africa', 'kenya'],
            ['winter', 'northern lights', 'high', 'europe', 'iceland'],
            ['autumn', 'wine tasting', 'high', 'south america', 'argentina'],
            ['spring', 'temples', 'low', 'asia', 'thailand'],
            ['summer', 'surfing', 'low', 'oceania', 'australia'],
            ['winter', 'shopping', 'medium', 'asia', 'dubai'],
            ['summer', 'surfing', 'medium', 'oceania', 'new zealand'],
            ['spring', 'hot springs', 'high', 'asia', 'japan'],
            ['winter', 'skiing', 'medium', 'north america', 'canada'],
            ['autumn', 'wine tasting', 'medium', 'europe', 'italy'],
            ['spring', 'museums', 'low', 'europe', 'greece'],
            ['summer', 'island hopping', 'medium', 'asia', 'philippines'],
            ['winter', 'shopping', 'high', 'asia', 'singapore'],
            ['autumn', 'hiking', 'low', 'south america', 'peru'],
            ['spring', 'temples', 'medium', 'asia', 'cambodia'],
            ['summer', 'beach', 'high', 'north america', 'bahamas'],
            ['winter', 'northern lights', 'medium', 'north america', 'alaska'],
            ['autumn', 'hot springs', 'low', 'europe', 'hungary'],
            ['spring', 'safari', 'high', 'africa', 'south africa'],
            ['summer', 'wine tasting', 'medium', 'south america', 'chile'],
            ['winter', 'museums', 'medium', 'europe', 'germany'],
            ['autumn', 'shopping', 'medium', 'asia', 'south korea'],
            ['spring', 'surfing', 'low', 'oceania', 'fiji'],
            ['summer', 'temples', 'low', 'asia', 'vietnam'],
            ['autumn', 'spa', 'high', 'europe', 'sweden'],
            ['winter', 'beach', 'medium', 'africa', 'seychelles'],
            ['spring', 'hiking', 'medium', 'europe', 'switzerland'],
            ['summer', 'safari', 'medium', 'africa', 'tanzania'],
            ['autumn', 'museums', 'low', 'north america', 'washington dc'],
            ['spring', 'island hopping', 'medium', 'oceania', 'french polynesia'],
            ['winter', 'hot springs', 'high', 'north america', 'yellowstone'],
            ['summer', 'shopping', 'low', 'asia', 'malaysia']
        ]

        X = [row[:4] for row in self.data]
        y = [row[4] for row in self.data]

        self.encoders = [LabelEncoder() for _ in range(4)]
        X_encoded = np.array([
            [self.encoders[i].fit_transform([row[i] for row in X])[j] for i in range(4)]
            for j in range(len(X))
        ])

        self.label_encoder = LabelEncoder()
        y_encoded = self.label_encoder.fit_transform(y)

        self.model = DecisionTreeClassifier()
        self.model.fit(X_encoded, y_encoded)

        # Activity mapping (frontend activity -> real activity)
        self.activity_map = {
            'adventure': ['skiing', 'surfing', 'hiking', 'safari'],
            'relaxation': ['beach', 'wine tasting', 'hot springs', 'spa'],
            'sightseeing': ['museums', 'northern lights', 'temples', 'shopping'],
            'cultural': ['temples', 'museums', 'wine tasting'],
            'beach': ['beach', 'surfing', 'island hopping'],
            'nature': ['safari', 'hiking', 'northern lights']
        }

        # Mapping real activity to destination (optional helper)
        self.activity_to_destination = {
            row[1]: row[4] for row in self.data
        }

    def predict(self, data):
        try:
            # Use simplified input
            season = data['season']
            activity_category = data['activity']
            budget = data['budget']
            continent = data['continent']

            # Choose a matching real-world activity from the category
            possible_activities = self.activity_map.get(activity_category, [])
            if not possible_activities:
                return {'error': 'Unknown activity category'}

            chosen_activity = random.choice(possible_activities)

            # Predict using the chosen activity
            input_row = [season, chosen_activity, budget, continent]
            input_encoded = [
                self.encoders[i].transform([input_row[i]])[0] for i in range(4)
            ]

            pred_encoded = self.model.predict([input_encoded])[0]
            destination = self.label_encoder.inverse_transform([pred_encoded])[0]

            return {
                'destination': destination.title(),
                'activitySuggestion': f"{chosen_activity.title()} in {destination.title()}"
            }

        except (ValueError, KeyError) as e:
            return {'error': f'Invalid input: {e}'}